# 3 3 conditional corrections

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections

source_corrections = get_source_corrections(window_days=28, post_correction_free=True)

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
from utils import StataDiD
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

### Second correction conditional on first correction

In [ ]:
from utils import get_coefs, att_plot
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==1')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from pystata import stata
stata.run('testparm treated_periods_post*')

In [ ]:
from pystata import stata
stata.run('matrix list e(b)')

In [ ]:
ylims = (-0.6, 0.7)
stata.run(f"local plotopts xtitle(Mbar, size(large)) ytitle(95% Robust CI, size(large)) yscale(r({ylims[0]} {ylims[1]})) ylabel({ylims[0]}(0.2){ylims[1]},labsize(large)) xlabel(,labsize(large)) ciopts(lwidth(thick)) xsize(4) ysize(3)")

for i in range(1,5):
    print(f"Post-treatment period {i}: \n")
    vec = [str(1) if e==i else str(0) for e in range(1,5)]
    vec_str = ' \\ '.join(vec)
    stata.run(f"matrix l_vec = {vec_str}")
    stata.run("honestdid, l_vec(l_vec) pre(9/11) post(12/15) mvec(0.5(0.5)2) alpha(0.05) grid_lb(-3) grid_ub(3)")
    stata.run("honestdid, cached coefplot `plotopts'")
    stata.run(f"graph export outputs/figures/honest_did_posts_weekly_prior_1_{i}.pdf, replace")

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==1')
mean_agg_att_1, ll_agg_att_1, hl_agg_att_1 = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att_1, ll_agg_att_1, hl_agg_att_1, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_piror_1', ylims=ylims, legend_ncol=1, legend_title='Second correction')

### Third correction

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==2')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==2')
mean_agg_att_2, ll_agg_att_2, hl_agg_att_2 = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att_2, ll_agg_att_2, hl_agg_att_2, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_piror_2', ylims=ylims, legend_ncol=1, legend_title='Third correction')

### Fourth correction

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==3')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==3')
mean_agg_att_3, ll_agg_att_3, hl_agg_att_3 = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att_3, ll_agg_att_3, hl_agg_att_3, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_piror_3', ylims=ylims, legend_ncol=1, legend_title='Fourth correction')

### Fifth correction

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==4')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections==4')
mean_agg_att_4, ll_agg_att_4, hl_agg_att_4 = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att_4, ll_agg_att_4, hl_agg_att_4, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_piror_4', ylims=ylims, legend_ncol=1, legend_title='Fifth correction')

### 6+ corrections

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections>=5')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster', condition='prior_corrections>=5')
mean_agg_att_5, ll_agg_att_5, hl_agg_att_5 = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att_5, ll_agg_att_5, hl_agg_att_5, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_piror_5', ylims=ylims, legend_ncol=1, legend_title='$\geq 6$ corrections')

### All corrections together

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_all_corrections', ylims=ylims, legend_ncol=1, legend_title='All corrections')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from utils import FONT_SIZE, LW, MZ, vermilion

plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE
plt.subplots(layout='constrained')
estimates = [mean_agg_att_1, mean_agg_att_2, mean_agg_att_3, mean_agg_att_4, mean_agg_att_5]
lls = [ll_agg_att_1, ll_agg_att_2, ll_agg_att_3, ll_agg_att_4, ll_agg_att_5]
hls = [hl_agg_att_1, hl_agg_att_2, hl_agg_att_3, hl_agg_att_4, hl_agg_att_5]
xticks = ['2', '3', '4', '5', '$\geq$6']
plt.axhline(y=0, linestyle='--', color='lightgrey', linewidth=LW)
for i in range(len(estimates)):
    plt.errorbar(xticks[i], estimates[i][0], yerr=[[estimates[i][0]-lls[i][0]], [hls[i][0]-estimates[i][0]]],
                 marker='_', linestyle='', color=vermilion, markersize=MZ, mew=LW, linewidth=LW)
plt.xticks(np.arange(len(xticks)), xticks)
plt.xlim(-0.25, len(xticks)-0.75)
plt.ylim(ylims)
plt.xlabel('Subsequent focal corrections')
plt.ylabel('Estimated DiD effect')
plt.savefig('outputs/figures/did_posts_weekly_prior_agg.pdf')
plt.show()